# Latent Dirichlet Allocation (LDA) — scikit-learn (fetch_20newsgroups)

Ce notebook montre comment effectuer du topic modeling avec `sklearn` en utilisant le jeu de données `fetch_20newsgroups`.
Nous incluons un exemple de base puis un réglage d'hyperparamètres via `optuna`.

In [ ]:
%pip install -q optuna pyLDAvis nltk wordcloud

In [ ]:
# Imports de base
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
import numpy as np
import optuna
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Charger le jeu de données (subset 'train' pour rapidité)
data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
texts = data.data
len(texts)

In [ ]:
# Split train / held-out pour évaluer la perplexité
train_texts, test_texts = train_test_split(texts, test_size=0.2, random_state=42)

# Vectorisation (counts) nécessaire pour LDA de sklearn
vectorizer = CountVectorizer(stop_words='english', max_df=0.95, min_df=2, max_features=10000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)
feature_names = vectorizer.get_feature_names_out()
X_train.shape

In [ ]:
# Baseline: entraînement d'un LDA simple
n_topics = 10
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method='online', max_iter=10)
lda.fit(X_train)
print('Perplexity (train):', lda.perplexity(X_train))
print('Perplexity (test):', lda.perplexity(X_test))

In [ ]:
# Fonction d'affichage des topics
def display_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f'Topic {topic_idx}: ' + ' '.join(top_words))

display_topics(lda, feature_names, n_top_words=12)

## Réglage des hyperparamètres avec Optuna
Nous allons optimiser le nombre de topics (`n_components`) et quelques paramètres de vectorisation et d'entraînement. L'objectif ici est de minimiser la perplexité sur l'ensemble tenu à l'écart (held-out).

In [ ]:
# Préparer les textes globaux (définis plus haut) pour la closure de l'objectif
# train_texts et test_texts existent déjà

def objective(trial):
    # Vectorizer params
    max_df = trial.suggest_float('max_df', 0.7, 0.99)
    min_df = trial.suggest_int('min_df', 1, 5)
    max_features = trial.suggest_categorical('max_features', [1000, 2000, 5000, 10000])

    vectorizer = CountVectorizer(stop_words='english', max_df=max_df, min_df=min_df, max_features=max_features)
    X_tr = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(test_texts)

    # LDA params
    n_components = trial.suggest_int('n_components', 5, 30)
    max_iter = trial.suggest_int('max_iter', 5, 30)
    learning_decay = trial.suggest_float('learning_decay', 0.5, 1.0)
    learning_offset = trial.suggest_int('learning_offset', 10, 100)

    lda = LatentDirichletAllocation(n_components=n_components, max_iter=max_iter, learning_method='online', learning_decay=learning_decay, learning_offset=learning_offset, random_state=42)
    lda.fit(X_tr)
    # Perplexity sur l'ensemble de validation (plus bas = meilleur)
    perp = lda.perplexity(X_val)
    return perp

# Lancer l'optimisation (réduire n_trials si vous avez peu de temps)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('Best value (perplexity):', study.best_value)
print('Best params:')
for k,v in study.best_params.items():
    print(' ', k, ':', v)

In [ ]:
# Entraîner le modèle final avec les meilleurs paramètres
best = study.best_params
vectorizer_final = CountVectorizer(stop_words='english', max_df=best.get('max_df', 0.95), min_df=best.get('min_df', 2), max_features=best.get('max_features', 10000))
X_train_final = vectorizer_final.fit_transform(train_texts)
X_test_final = vectorizer_final.transform(test_texts)
lda_final = LatentDirichletAllocation(n_components=best.get('n_components',10), max_iter=best.get('max_iter',10), learning_method='online', learning_decay=best.get('learning_decay',0.7), learning_offset=best.get('learning_offset',50), random_state=42)
lda_final.fit(X_train_final)
print('Final perplexity (test):', lda_final.perplexity(X_test_final))
display_topics(lda_final, vectorizer_final.get_feature_names_out(), n_top_words=12)

### Visualisation par Nuages de Mots (Word Clouds)
Les word clouds permettent de voir rapidement les termes les plus importants pour chaque topic.

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_word_clouds(model, feature_names, n_topics=5, words_per_topic=20):
    # On sélectionne les n_topics premiers ou tous s'il y en a moins
    n_topics = min(n_topics, model.n_components)
    cols = 2
    rows = (n_topics + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
    axes = axes.flatten()

    for i in range(n_topics):
        # Récupérer les poids des mots pour ce topic
        topic_weights = model.components_[i]
        # Créer un dictionnaire {mot: poids}
        word_freq = {feature_names[j]: topic_weights[j] for j in topic_weights.argsort()[:-words_per_topic - 1:-1]}
        
        wc = WordCloud(background_color='white', width=400, height=300).generate_from_frequencies(word_freq)
        
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(f'Topic {i}', fontsize=16)
        axes[i].axis('off')

    # Cacher les axes vides si nécessaire
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# Affichage des nuages de mots pour le modèle final
plot_word_clouds(lda_final, vectorizer_final.get_feature_names_out(), n_topics=lda_final.n_components)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_topic_distribution_sample(model, X, n_samples=20):
    # Récupérer la distribution des topics pour les documents
    doc_topic_dist = model.transform(X[:n_samples])
    
    # Créer un DataFrame pour faciliter l'affichage
    df = pd.DataFrame(doc_topic_dist, 
                      columns=[f'Topic {i}' for i in range(model.n_components)])
    df.index = [f'Doc {i}' for i in range(n_samples)]
    
    # Affichage
    plt.figure(figsize=(14, 8))
    df.plot(kind='bar', stacked=True, colormap='tab20', ax=plt.gca())
    
    plt.title(f'Composition des {n_samples} premiers documents par Topic', fontsize=16)
    plt.xlabel('Documents')
    plt.ylabel('Proportion du Topic')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Topics")
    plt.tight_layout()
    plt.show()

# Visualisation pour les 25 premiers documents du dataset final
plot_topic_distribution_sample(lda_final, X_train_final, n_samples=25)